# Go2 Track 2: PyTorch PPO High-Level Planner

这个 notebook 用固定低层 Go2 checkpoint 训练高层 `ppo_mlp` planner。高层 actor 输入官方 5D track observation，输出 `[vx, vy, yaw_rate]`。

## 1. 进入项目目录

先把项目上传或 clone 到 Colab。下面默认项目在 `/content/Final-Project-Track-2-Bonus-Project`。

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/Final-Project-Track-2-Bonus-Project')
assert PROJECT_DIR.exists(), f'Project directory not found: {PROJECT_DIR}'
%cd /content/Final-Project-Track-2-Bonus-Project

## 2. 安装依赖

Colab 通常已经预装 PyTorch。MuJoCo/MJX/Brax 等依赖按仓库 requirements 安装。

In [ ]:
!pip install -q -r configs/colab_requirements.txt

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

## 3. 指定低层 checkpoint

本路线不重新训练 low-level。把已有 Brax PPO checkpoint 目录填到 `CHECKPOINT_DIR`。

In [ ]:
CHECKPOINT_DIR = 'best_checkpoint'

# 如果你上传的是 zip，可以先解压，再把 CHECKPOINT_DIR 改成解压后的真正 checkpoint 目录。
# !unzip -q hw1_best_checkpoint.zip -d hw1_best_checkpoint_unzipped

checkpoint_path = Path(CHECKPOINT_DIR)
print('checkpoint exists:', checkpoint_path.exists())
print('metadata exists:', (checkpoint_path / 'ppo_network_config.json').exists())

## 4. 快速检查低层 policy

这一步只跑很短 rollout，用来确认 checkpoint 能被恢复。

In [ ]:
!python quick_policy_check.py --checkpoint-dir "{CHECKPOINT_DIR}" --num-steps 100

## 5. 短调试训练

先用小并行数确认 PPO/JAX 桥接能跑通。

In [ ]:
!python train_highlevel_ppo_torch.py \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --output-dir artifacts/highlevel_ppo_torch_debug \
  --total-updates 3 \
  --num-envs 16 \
  --rollout-steps 128 \
  --eval-interval 0

## 6. 正式训练

如果显存不足，把 `--num-envs` 降到 16 或 32，把 `--rollout-steps` 降到 128。

In [ ]:
!python train_highlevel_ppo_torch.py \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --output-dir artifacts/highlevel_ppo_torch \
  --total-updates 80 \
  --num-envs 64 \
  --rollout-steps 256 \
  --ppo-epochs 4 \
  --minibatch-size 1024 \
  --start-randomization curriculum \
  --eval-interval 10 \
  --eval-seconds 120

## 7. 官方式评估

先 no-render 快速看指标；确认不错后去掉 `--no-render` 生成视频。

In [ ]:
!python run_track_bonus.py \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --planner-config artifacts/highlevel_ppo_torch/planner_config.json \
  --output-dir artifacts/highlevel_ppo_torch_eval \
  --duration-seconds 300 \
  --entry-name ppo_mlp \
  --no-render

In [ ]:
import json

result_path = Path('artifacts/highlevel_ppo_torch_eval/results.json')
payload = json.loads(result_path.read_text())
payload['metrics'], payload['scores']

## 8. 打包训练产物

提交时至少需要 planner config、planner weights、修改后的 planner code、训练脚本、低层 checkpoint 和评估结果。

In [ ]:
!zip -qr highlevel_ppo_submission_bundle.zip \
  artifacts/highlevel_ppo_torch/planner_config.json \
  artifacts/highlevel_ppo_torch/planner_weights.npz \
  artifacts/highlevel_ppo_torch/training_history.json \
  artifacts/highlevel_ppo_torch_eval/results.json \
  train_highlevel_ppo_torch.py \
  track_bonus/planner.py \
  docs/ppo_high_level_route.md

print('bundle ready: highlevel_ppo_submission_bundle.zip')